# KRAS: learning to compare two groups

This notebook explains what is measured, what is compared, and how to interpret the result. Run cells from top to bottom. The original `simpleHypo_analys.ipynb` remains a working draft.

**Question:** does the mean KRAS knockout effect differ between cell lines with a recorded KRAS hotspot and lines without one?

Each row represents one cell model. The outcome is KRAS Gene Effect: a more negative score corresponds to a stronger reduction in growth or viability. It is not the probability of cell death. Expression is not part of this comparison yet; it will be added when moving to prediction.

Analysis path: validate data → describe groups → inspect distributions → estimate the mean difference → run the test → assess sensitivity to related lines and group definitions → state a limited conclusion.


## 1. Load data and tools

**Why:** make the analysis reproducible from a saved CSV without variables from another notebook. pandas handles tables, NumPy numbers, SciPy distributions and tests, and matplotlib plots.

If needed, install dependencies in the selected kernel with `%pip install pandas numpy scipy matplotlib`. The next cell supports execution from either the pilot directory or the project root.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "data" / "pilot"]
pilot_dir = next((p for p in candidates if (p / "KRAS_data.csv").exists()), None)
if pilot_dir is None:
    raise FileNotFoundError("Run the notebook from data/pilot or the project root")

raw = pd.read_csv(pilot_dir / "KRAS_data.csv")
models = pd.read_csv(pilot_dir.parent / "COAD_Model.csv")
print("File:", pilot_dir / "KRAS_data.csv")
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__, "| scipy:", scipy.__version__)
print("Shape:", raw.shape)
display(raw.head())

## 2. Verify what the groups mean

**Why:** a statistical test cannot repair incorrectly defined groups.

- `KRAS_hotspot = 1`: at least one KRAS hotspot variant is recorded in the selected profile.
- `KRAS_hotspot = 0`: no hotspot was found. The line may still carry another KRAS mutation.
- Missing: status is unknown; the row must not automatically enter group 0.

Therefore, replace “with Mutation” and “without Mutation” with “with hotspot” and “without hotspot.” Verify the flag against the source variant table. A complete absence of mutation records remains unknown. If KRAS hotspot annotations are unknown and no hotspot is confirmed, status also remains unknown.


In [ ]:
assert raw["ModelID"].notna().all() and raw["ModelID"].is_unique
assert raw["KRAS_hotspot"].dropna().isin([0, 1]).all()

mut = pd.read_csv(
    pilot_dir.parent / "COAD_Mutations.csv",
    usecols=["ModelID", "IsDefaultEntryForModel", "HugoSymbol", "Hotspot"],
    low_memory=False
)
mut = mut.loc[mut["IsDefaultEntryForModel"].eq("Yes")].copy()
kras = mut.loc[mut["HugoSymbol"].eq("KRAS")].copy()
kras["hotspot_flag"] = (
    kras["Hotspot"].astype("string").str.strip().str.lower()
    .map({"true": True, "false": False}).astype("boolean")
)
positive_ids = kras.loc[kras["hotspot_flag"].fillna(False), "ModelID"]
unknown_ids = kras.loc[kras["hotspot_flag"].isna(), "ModelID"]

data = raw.merge(
    models[["ModelID", "CellLineName", "PatientID"]],
    on="ModelID", how="left", validate="one_to_one"
)
expected = data["ModelID"].isin(positive_ids).astype("Int64")
unknown = (
    ~data["ModelID"].isin(mut["ModelID"])
    | (data["ModelID"].isin(unknown_ids) & ~data["ModelID"].isin(positive_ids))
)
expected.loc[unknown] = pd.NA
old = data["KRAS_hotspot"].astype("Int64")
changed = (old.fillna(-1) != expected.fillna(-1)).sum()
print("Statuses corrected after verification:", changed)
data["KRAS_hotspot"] = expected
data["has_KRAS_variant_record"] = data["ModelID"].isin(kras["ModelID"])

eligible = data["KRAS_hotspot"].notna() & data["KRAS_GeneEffect"].notna()
print("Rows excluded because status or y is unknown:", int((~eligible).sum()))
analysis = data.loc[eligible].copy()
assert np.isfinite(analysis["KRAS_GeneEffect"]).all()
analysis["group"] = analysis["KRAS_hotspot"].map({1: "With hotspot", 0: "Without hotspot"})
display(analysis.groupby("group")["has_KRAS_variant_record"].agg(["size", "sum"]))

## 3. Check observation independence

**Why:** standard two-sample t-tests assume independent observations. Two lines from the same patient can be similar because they share an origin and are not fully independent replicates.

Add PatientID only to inspect data structure, not as a numerical feature. An unknown PatientID receives a separate technical group, which does not prove independence. The primary calculation using all rows is preliminary. It is repeated below with one line per patient.


In [ ]:
analysis["patient_group"] = analysis["PatientID"].fillna(analysis["ModelID"])
related = analysis.loc[analysis["patient_group"].duplicated(keep=False)]
print("Cell lines:", len(analysis))
print("PatientID groups:", analysis["patient_group"].nunique())
print("Unknown PatientID:", analysis["PatientID"].isna().sum())
display(related[["ModelID", "CellLineName", "PatientID", "group", "KRAS_GeneEffect"]])

## 4. Describe the data before testing

**Why:** understand effect size and spread instead of starting with a single p-value.

- count — number of cell lines;
- mean — the quantity compared by the t-test;
- std — standard deviation of individual observations;
- 50% — median; 25% and 75% bound the middle half;
- min/max — extreme values.

Define the difference as **mean with hotspot minus mean without hotspot**. A negative value supports the expected direction. The difference is measured in Gene Effect units; negative means must not be divided to claim an “x-fold greater lethality.”


In [ ]:
with_hotspot = analysis.loc[analysis["KRAS_hotspot"].eq(1), "KRAS_GeneEffect"]
without_hotspot = analysis.loc[analysis["KRAS_hotspot"].eq(0), "KRAS_GeneEffect"]
assert min(len(with_hotspot), len(without_hotspot)) >= 2
display(analysis.groupby("group")["KRAS_GeneEffect"].describe())
difference = with_hotspot.mean() - without_hotspot.mean()
variance_ratio = with_hotspot.var(ddof=1) / without_hotspot.var(ddof=1)
print(f"Mean difference: {difference:.3f}")
print(f"Sample variance ratio: {variance_ratio:.2f}")

## 5. Plot every cell line

**Why:** reveal group overlap, asymmetry, and extreme observations. Bars showing only means would hide this information.

Each point is a cell line. Small random horizontal jitter only separates points and does not alter Gene Effect. The box shows quartiles and its inner line is the median. Zero is a scale reference, not a boundary of proven lethality.


In [ ]:
rng = np.random.default_rng(42)
fig, ax = plt.subplots(figsize=(8, 5))
arrays = [without_hotspot.to_numpy(), with_hotspot.to_numpy()]
ax.boxplot(arrays, positions=[0, 1], widths=0.4, showfliers=False)
for position, values in enumerate(arrays):
    ax.scatter(position + rng.uniform(-0.12, 0.12, len(values)), values,
               alpha=0.75, s=35)
ax.set_xticks([0, 1], ["Without hotspot", "With hotspot"])
ax.set_ylabel("KRAS Gene Effect (lower means a stronger knockout effect)")
ax.axhline(0, color="grey", linestyle="--", linewidth=1)
ax.set_title("Distributions overlap even when means differ")
fig.tight_layout()
plt.show()

## 6. What the t-test evaluates

**Null hypothesis H₀:** mean Gene Effect is equal in the two groups. **Alternative H₁:** the means differ.

Use a two-sided test, which permits a difference in either direction. A negative difference is expected, but the test must not be changed to one-sided after seeing the result.

The p-value is the probability, under H₀ and the test assumptions, of obtaining a t-statistic at least as extreme as the observed one. It is **not** the probability that H₀ is true, the probability that the whole study is random, or the probability of successful prediction.

The 0.05 threshold is a decision convention, not a boundary of biological importance. This comparison was specified in advance. Testing many genes and variants would require correction for multiple comparisons.


## 7. Why Welch instead of the ordinary Student test

Here, “Student test” means the two-sample test with a pooled variance estimate. Welch is also a member of the t-test family.

| Property | Student | Welch |
|---|---|---|
| Compared quantity | Means | Means |
| Equal population variances | Assumed | Not required |
| Independent observations | Required | Required |
| Degrees of freedom | n₁ + n₀ − 2 | Computed from group sizes and variances |

In the source summaries, standard deviations are approximately 0.73 and 0.39, so sample variances differ by about 3.6-fold. This is descriptive evidence, not proof that population variances differ. Since equal variances are not justified, Welch avoids that extra assumption. There is no need to test variance equality first and switch methods based on its p-value.

**Nuance:** with equal group sizes, Student and Welch t-statistics coincide. Degrees of freedom and p-values differ, so Student would not necessarily lead to the opposite conclusion.

Welch does not correct dependence between related lines, data errors, or severe outliers. The t-distribution justification is connected to normality; with moderate samples the test is often used approximately. With 23 rows, plots matter. The Q–Q plot below diagnoses shape rather than automatically permitting or prohibiting the test.

[SciPy documentation: equal_var=True / False](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_ind.html).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, values, title in zip(axes, arrays, ["Without hotspot", "With hotspot"]):
    stats.probplot(values, dist="norm", plot=ax)
    ax.set_title("Q–Q: " + title)
fig.tight_layout()
plt.show()

student = stats.ttest_ind(with_hotspot, without_hotspot, equal_var=True)
welch = stats.ttest_ind(with_hotspot, without_hotspot, equal_var=False)
display(pd.DataFrame({
    "Test": ["Student", "Welch"],
    "t": [student.statistic, welch.statistic],
    "df": [student.df, welch.df],
    "p_value": [student.pvalue, welch.pvalue]
}))

## 8. Build the Welch calculation step by step

**Why:** understand where t, the p-value, and the confidence interval come from.

1. Calculate group means and difference Δ.
2. Calculate sample variances s² with ddof=1, dividing by n−1.
3. Divide variance by n to estimate the variance of the mean. More independent observations produce a more precise mean.
4. Add the variances of the two means and take the square root: SE = √(s₁²/n₁ + s₀²/n₀). **SE is uncertainty in the difference**; std is dispersion among individual lines.
5. t = Δ/SE: the number of standard errors separating the observed difference from zero.
6. Welch–Satterthwaite degrees of freedom select the appropriate t-distribution and may be fractional.
7. Compute the two-tailed p-value.
8. Build the 95% interval: Δ ± critical t × SE. It gives differences compatible with the data under the method assumptions.

Frequentist interpretation of 95%: if the valid procedure were repeated, about 95% of such intervals would cover the true difference. This is not an interval containing 95% of individual cell lines.


In [ ]:
n1, n0 = len(with_hotspot), len(without_hotspot)
mean1, mean0 = with_hotspot.mean(), without_hotspot.mean()
s1_squared, s0_squared = with_hotspot.var(ddof=1), without_hotspot.var(ddof=1)
v1, v0 = s1_squared / n1, s0_squared / n0
delta = mean1 - mean0
se = np.sqrt(v1 + v0)
t_manual = delta / se
df_manual = (v1 + v0)**2 / (v1**2 / (n1 - 1) + v0**2 / (n0 - 1))
p_manual = 2 * stats.t.sf(abs(t_manual), df=df_manual)
critical = stats.t.ppf(0.975, df=df_manual)
ci_low, ci_high = delta - critical * se, delta + critical * se

display(pd.Series({
    "Mean difference": delta, "Standard error SE": se,
    "t": t_manual, "df": df_manual, "p-value": p_manual,
    "95% CI lower bound": ci_low, "95% CI upper bound": ci_high
}))
np.testing.assert_allclose(t_manual, welch.statistic)
np.testing.assert_allclose(p_manual, welch.pvalue)
print("The manual calculation matches SciPy.")

## 9. Check the influence of related cell lines

**Why:** HCT-15 and DLD-1 share a PatientID. The preliminary test on 46 rows does not account for this.

A simple educational sensitivity check retains one line per patient. The rule is chosen without looking at Gene Effect: first the minimum ModelID, then the maximum ModelID. Both results are shown rather than selecting the smaller p-value.

This reduces repeated representation of one patient but does not guarantee the absence of other dependencies or confounders. More advanced analyses can use models that account for patient groups.


In [ ]:
def compare_groups(frame, label):
    a = frame.loc[frame["KRAS_hotspot"].eq(1), "KRAS_GeneEffect"]
    b = frame.loc[frame["KRAS_hotspot"].eq(0), "KRAS_GeneEffect"]
    if min(len(a), len(b)) < 2:
        raise ValueError("At least two observations per group are required")
    result = stats.ttest_ind(a, b, equal_var=False)
    ci = result.confidence_interval(confidence_level=0.95)
    return {
        "Scenario": label, "n_hotspot": len(a), "n_no_hotspot": len(b),
        "Difference": a.mean() - b.mean(),
        "CI_low": ci.low, "CI_high": ci.high, "p_value": result.pvalue
    }

ordered = analysis.sort_values("ModelID")
one_per_patient = ordered.drop_duplicates("patient_group", keep="first")
alternative_patient = ordered.drop_duplicates("patient_group", keep="last")
sensitivity = pd.DataFrame([
    compare_groups(analysis, "All lines: preliminary"),
    compare_groups(one_per_patient, "One per patient: minimum ModelID"),
    compare_groups(alternative_patient, "One per patient: maximum ModelID")
])
display(sensitivity)

## 10. Check the definition of the “without hotspot” group

**Why:** the original biological idea compared hotspots against the absence of a recorded KRAS variant. However, zero hotspot also includes other variants.

Show those lines and repeat the analysis without them, again using one line per patient. This is a sensitivity analysis with a slightly different comparison group. It must not silently replace the primary result merely because its p-value is more convenient.

The absence of a recorded variant still does not prove wild type; additional coverage and analysis information would be required.


In [ ]:
other_variants = analysis["KRAS_hotspot"].eq(0) & analysis["has_KRAS_variant_record"]
display(analysis.loc[other_variants, [
    "ModelID", "CellLineName", "KRAS_protein_changes", "KRAS_GeneEffect"
]])
strict = (
    analysis.loc[~other_variants].sort_values("ModelID")
    .drop_duplicates("patient_group", keep="first")
)
display(pd.DataFrame([
    compare_groups(one_per_patient, "Without hotspot: includes other variants"),
    compare_groups(strict, "Without any recorded KRAS variants")
]))

## 11. Check whether one cell line determines the result

**Why:** a substantial difference may depend on one extreme observation. Remove one row at a time from the one-line-per-patient dataset. No rows are deleted from the source CSV.

This is a robustness diagnostic, not a new independent experiment or a reason to select the smallest p-value. Focus primarily on the range of mean differences.


In [ ]:
leave_one_out = pd.DataFrame([
    {"Excluded": row["CellLineName"], **compare_groups(
        one_per_patient.drop(index=index), "One line excluded"
    )}
    for index, row in one_per_patient.iterrows()
])
print("Difference when excluding one line:",
      leave_one_out["Difference"].min(), "…", leave_one_out["Difference"].max())
display(leave_one_out.sort_values("Difference").iloc[[0, -1]])

## 12. Writing the conclusion and choosing the next step

A conclusion should report group sizes, direction and magnitude of the difference, confidence interval, and limitations. “p < 0.05” alone is insufficient. The summary below uses one line per patient and the minimum ModelID rule.

The statistical difference indicates an association between hotspot status and CRISPR response in these data. It does not prove that the mutation alone causally explains the difference because the groups may differ in other molecular properties. It also does not mean every hotspot makes a cell dependent or that patients would show the same effect.

**The next step is a separate predictive task:** hide outcomes for part of the models, train a hotspot-based predictor on the rest, compare it with the overall training mean, and evaluate error on hidden outcomes. Then add expression. Patient groups must not be split between train and test. This notebook tests group differences and does not train an ML model.


In [ ]:
report = compare_groups(one_per_patient, "Primary sensitivity analysis")
print(
    f"One line per patient: {report['n_hotspot']} with hotspot and "
    f"{report['n_no_hotspot']} without hotspot.\n"
    f"Mean difference (with − without) = {report['Difference']:.3f}; "
    f"95% CI [{report['CI_low']:.3f}; {report['CI_high']:.3f}]; "
    f"two-sided p = {report['p_value']:.3g}."
)
if report["CI_high"] < 0:
    print("The data support a more negative mean KRAS knockout effect in the hotspot group.")
elif report["CI_low"] > 0:
    print("The difference points in the opposite direction: the mean effect is higher in the hotspot group.")
else:
    print("The interval includes zero: the data are insufficient for a confident conclusion about a mean difference.")
print("This is a group comparison, not an estimate of predictive accuracy for a new cell line.")